In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from tqdm import tqdm


In [ ]:
VALIDATION_LOOP_DIR = Path(".").resolve()
REPO_DIR = VALIDATION_LOOP_DIR.parent

sys.path.insert(0, str((REPO_DIR / "training_loop").resolve()))
sys.path.insert(0, str(REPO_DIR.resolve()))

DATASET_DIR = REPO_DIR.parent / "dataset"
TRAIN_DIR = DATASET_DIR / "Dataset_train"
VAL_DIR = DATASET_DIR / "Dataset_validation"
TRAIN_CSV = REPO_DIR / "tags" / "train.csv"
VAL_CSV = REPO_DIR / "tags" / "validation.csv"
OUTPUT_DIR = REPO_DIR / "output" / "history_3d_residual_cnn_1"

label_columns = ["ICH"]
target_size = 128

epoch = 8
batch_size_train = 8
batch_size_val = 8
num_workers = 4


In [ ]:
from dataset import CTVolumeDataset


class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout3d(p=dropout_p) if dropout_p > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()

        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(out_channels)
        self.dropout = nn.Dropout3d(p=dropout_p) if dropout_p > 0 else nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.InstanceNorm3d(out_channels),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)

        out = F.relu(self.norm1(self.conv1(x)), inplace=True)
        out = self.dropout(out)
        out = self.norm2(self.conv2(out))
        out = F.relu(out + identity, inplace=True)
        return out


class Residual3DClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()

        self.encoder = nn.Sequential(
            ConvBlock3D(in_channels, 16, stride=2, dropout_p=0.1),
            ResidualBlock3D(16, 16, stride=1, dropout_p=0.1),
            ResidualBlock3D(16, 32, stride=2, dropout_p=0.1),
            ResidualBlock3D(32, 32, stride=1, dropout_p=0.1),
            ResidualBlock3D(32, 64, stride=2, dropout_p=0.1),
            ResidualBlock3D(64, 64, stride=1, dropout_p=0.1),
            ResidualBlock3D(64, 128, stride=2, dropout_p=0.1),
            ResidualBlock3D(128, 128, stride=1, dropout_p=0.1),
            ResidualBlock3D(128, 256, stride=2, dropout_p=0.1),
            ResidualBlock3D(256, 256, stride=1, dropout_p=0.1),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.head(x)
        return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_dataset = CTVolumeDataset(
    table_path=TRAIN_CSV,
    images_dir=TRAIN_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

val_dataset = CTVolumeDataset(
    table_path=VAL_CSV,
    images_dir=VAL_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_val,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)

model = Residual3DClassifier(in_channels=1, num_classes=len(label_columns)).to(device)
checkpoint = torch.load(OUTPUT_DIR / f"checkpoint_epoch_{epoch}.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


In [ ]:
def collect_split_predictions(model, loader, dataset, device, split_name):
    study_uids = []
    targets = []
    scores = []

    with torch.no_grad():
        for batch_idx, (volumes, labels) in enumerate(tqdm(loader, desc=f"Predict {split_name}", leave=False)):
            volumes = volumes.to(device, non_blocking=True)
            logits = model(volumes)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)

            start = batch_idx * loader.batch_size
            end = start + len(labels)
            study_uids.extend(dataset.samples_df.iloc[start:end]["study_uid"].astype(str).tolist())
            scores.append(probs)
            targets.append(labels.numpy().reshape(-1))

    y_true = np.concatenate(targets, axis=0)
    y_score = np.concatenate(scores, axis=0)
    predictions_df = pd.DataFrame(
        {
            "study_uid": study_uids,
            "target": y_true.astype(int),
            "predict": y_score,
        }
    )
    return y_true, y_score, predictions_df


train_y_true, train_y_score, train_predictions = collect_split_predictions(
    model=model,
    loader=train_loader,
    dataset=train_dataset,
    device=device,
    split_name="train",
)

val_y_true, val_y_score, val_predictions = collect_split_predictions(
    model=model,
    loader=val_loader,
    dataset=val_dataset,
    device=device,
    split_name="validation",
)


In [ ]:
def plot_preliminary_plots(y_true, y_score, dataset_name):
    fig_hist = px.histogram(x=y_score, color=y_true.astype(str), nbins=50,
                            labels={'color': 'True Labels', 'x': 'Score'},
                            title=f'{dataset_name}: Histogram of Scores',
                            width=800,
                            height=500)
    fig_hist.show()

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    df_thresh = pd.DataFrame({'FPR': fpr, 'TPR': tpr}, index=thresholds)
    
    fig_thresh = px.line(df_thresh, 
                         title=f'{dataset_name}: TPR and FPR at every threshold',
                         width=800,
                         height=500)
    fig_thresh.update_yaxes(scaleanchor="x", scaleratio=1)
    fig_thresh.update_xaxes(range=[0, 1], constrain='domain')
    fig_thresh.show()

def plot_roc_curve(y_true, y_score, dataset_name):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    
    fig = px.area(x=fpr, y=tpr, 
                  title=f'{dataset_name}: ROC Curve (AUC={roc_auc:.3f})',
                  labels={'x': 'False Positive Rate', 'y': 'True Positive Rate'},
                  width=800,
                  height=500)
    fig.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()

def plot_pr_curve(y_true, y_score, dataset_name):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)
    
    fig = px.area(x=recall, y=precision, 
                  title=f'{dataset_name}: Precision-Recall Curve (AUC={pr_auc:.3f})',
                  labels={'x': 'Recall', 'y': 'Precision'},
                  width=800,
                  height=500)
    fig.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=1, y1=0)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()


In [ ]:
for y_true, y_score, name in [(train_y_true, train_y_score, 'Train'),
                               (val_y_true, val_y_score, 'Validation')]:
    plot_preliminary_plots(y_true, y_score, name)
    plot_roc_curve(y_true, y_score, name)
    plot_pr_curve(y_true, y_score, name)


In [ ]:
train_predictions.to_csv(OUTPUT_DIR / "predict_train_3d_residual_cnn_1.csv", index=False)
val_predictions.to_csv(OUTPUT_DIR / "predict_val_3d_residual_cnn_1.csv", index=False)
